In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter # Split doc by chunck
from langchain_openai import OpenAIEmbeddings, ChatOpenAI # Open AI model to convert text to vector
from langchain_chroma import Chroma # Vector db
from langchain_community.document_loaders import PyPDFLoader # Load PDF file
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

In [23]:
import os
from dotenv import load_dotenv

In [24]:
# Load .env
load_dotenv()

# Get API key
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

# Model Loading (Embeddings & LLM)
embeddings_model = OpenAIEmbeddings()
llm = ChatOpenAI(model_name = 'gpt-3.5-turbo', max_tokens = 200)

In [25]:
# Load PDF
pdf_path = 'rag_file_example.pdf'

loader = PyPDFLoader(pdf_path, extract_images = False)
pages = loader.load_and_split()

In [26]:
# Chunk Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 4000,
    chunk_overlap = 20, # Maintain context between chunks
    length_function = len,
    add_start_index = True,
)

chunks = text_splitter.split_documents(pages)

In [27]:
# Store on Vector DB - Chroma
data_base = Chroma()

# Store all chuncks with defined embedding on correctly directory
data_base.from_documents(
    chunks,
    embedding = embeddings_model,
    persist_directory = 'text_index',
)

In [28]:
# Load Vector DB
vector_db = Chroma(
    persist_directory = 'text_index',
    embedding_function = embeddings_model,
)

# Load Retriever
retriever = vector_db.as_retriever(search_kwargs = {'k': 3}) # Get top 3 important docs

# Build prompt chain to call LLM
prompt = ChatPromptTemplate.from_template("""
    Você é um assistente técnico. Use apenas o contexto fornecido para responder.
    Se a resposta não estiver no contexto, diga que não sabe.
    
    Contexto:
    {context}
    
    Pergunta: {input}
""")
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
chain = create_retrieval_chain(retriever, combine_docs_chain)

In [29]:
def ask(question):
    response = chain.invoke({"input": question})
    return response["answer"]

In [30]:
user_question = input('User Question: ')
answer = ask(user_question)
print(answer)

aqui


User Question:  Quais os principais pontos de atenção da LEI?


Os principais pontos de atenção da Lei são a necessidade de informar claramente as pessoas afetadas por sistemas de inteligência artificial sobre o caráter automatizado da interação e decisão, a descrição geral do sistema, a identificação dos operadores, o papel dos humanos envolvidos, as categorias de dados pessoais utilizados, as medidas de segurança, não-discriminação e confiabilidade adotadas, além de outras informações definidas em regulamento. É essencial fornecer essas informações de forma clara e adequada antes da contratação ou utilização do sistema.
